In [1]:
#import libraries
import sys
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Make the repo root importable so `from src.data_loader import load_cases` works
# regardless of where Jupyter was launched from.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data_loader import load_cases

In [2]:
# load the full merged dataset (both shards) via the shared loading pipeline.
# First run builds+caches data/processed/all_cases.parquet; later runs just read the cache.
df = load_cases()
print(df.shape)

(33610, 20)


In [3]:
# sanity check: columns and case_type breakdown (hearing + appeal, both shards)
print(df.columns)
print(df["case_type"].value_counts())

Index(['row_id', 'case_number', 'date', 'outcome', 'guidelines', 'summary',
       'full_text', 'sor_allegations', 'mitigating_factors', 'judge',
       'source_url', 'formal_findings', 'case_type', 'appeal_board_members',
       'who_appealed', 'judges_findings_of_fact', 'judges_analysis',
       'discussion', 'order', 'source_shard'],
      dtype='str')
case_type
hearing    27973
appeal      5637
Name: count, dtype: int64


In [4]:
# drop any rows missing full_text/outcome (currently a no-op safety guard, not a cleaning
# pass -- see markdown note below), randomize, and take a 2000-case sample for this baseline
df = df.dropna(subset=['full_text', 'outcome']).sample(frac=1, random_state=42)
df = df.head(2000)
print(df.shape)
print(df['outcome'].value_counts())

(2000, 20)
outcome
DENIED      1263
GRANTED      536
UNKNOWN      174
REMANDED      26
REVOKED        1
Name: count, dtype: int64


### ⚠️ Data-quality note for EDA (not fixed here on purpose)

The `.dropna(subset=['full_text', 'outcome'])` above is a **no-op safety guard** right now, not a
real cleaning pass — as of this dataset, `full_text` and `outcome` have 0 nulls across all 33,610
rows (there are 2 rows with an *empty string* `full_text`, which `dropna` wouldn't catch anyway).
It's kept here defensively in case that changes, not as a substitute for an actual missing-data
strategy.

More importantly, this cell does **not** attempt to fix any of the known data-quality issues
documented in [`data/README.md`](../data/README.md) — duplicate rows, ambiguous re-decisions,
unparsed/messy dates, empty-string text, mojibake, etc. Those are EDA decisions for the team to
make together (e.g. dedup strategy, how to treat empty-string cases), not something a baseline
script should quietly decide on everyone's behalf.

**TODO (EDA):** review `data/README.md`'s data-quality quirks and agree on a cleaning strategy
before it's applied anywhere beyond this quick, unfiltered baseline.

### ⚠️ Data-quality note for EDA (not fixed here on purpose)

Now that this notebook loads the **full merged dataset** via `load_cases()` (33,610 rows, both
shards) instead of just `all_cases_1.parquet`, the `outcome` counts above can include very rare
classes — e.g. `REVOKED` has only ~15 cases across the *entire* dataset, so a fixed 2,000-row
sample can draw just 0 or 1 of them. That's too few for a **stratified** train/val split
(`train_test_split(..., stratify=...)` requires ≥2 examples per class in the split).

We deliberately did **not** drop/merge rare classes here — how to handle them (drop, bucket into
"OTHER", oversample, etc.) is a data-cleaning decision that should be made during EDA, together
with the team, not silently inside a baseline script. For now, the split below just uses a plain
random split (no `stratify=`) so the notebook runs regardless of which rare classes happen to land
in the sample.

**TODO (EDA):** decide how to handle rare `outcome` classes (see `data/README.md` for the full
outcome distribution) before finalizing the real train/val/test split.

In [5]:
# split the text into training and validation sets
# NOTE: no stratify= here -- see markdown note above re: rare outcome classes (e.g. REVOKED)
X_train, X_val, y_train, y_val = train_test_split(
    df['full_text'], 
    df['outcome'], 
    test_size=0.2, 
    random_state=42,
)

print("Training cases:", len(X_train))
print("Validation cases:", len(X_val))

Training cases: 1600
Validation cases: 400


In [6]:
# create a TF-IDF vectorizer
vec = TfidfVectorizer(max_features=2000, ngram_range=(1, 2))

Xtr = vec.fit_transform(X_train) #x training data
Xv = vec.transform(X_val)   # x validation data

print("Training TF-IDF shape:", Xtr.shape)
print("Validation TF-IDF shape:", Xv.shape)

Training TF-IDF shape: (1600, 2000)
Validation TF-IDF shape: (400, 2000)
